# Optional Exercise Solution: End-to-End Classification Workflow


This notebook is one possible solution to the optional exercise. We use the `water-quality.csv` dataset, build a classification model, evaluate it with a suitable metric, and then improve it with hyperparameter tuning.

Many other valid solutions are possible. The important part is the workflow: choose a model deliberately, evaluate it with the right metric, and compare the tuned model against a baseline.


The plan for this solution is:

```mermaid
flowchart LR
    A[Load data] --> B[Inspect target]
    B --> C[Build baseline pipeline]
    C --> D[Evaluate baseline]
    D --> E[Tune hyperparameters]
    E --> F[Compare results]
```


## Choose the Dataset and Metric


We choose the water-quality dataset. The target column `is_safe` tells us whether a water sample is safe to drink, so this is a binary classification problem.

Because the classes are imbalanced, we use **F1 score** as the main evaluation metric. Accuracy alone would hide whether the model performs well on the minority class.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("data/water-quality.csv")
df.head()

In [ ]:
df["is_safe"].value_counts()

The positive class is less frequent, which is exactly the kind of situation where precision, recall, and F1 are often more informative than accuracy.


## Build a Baseline Model


We use a logistic regression model inside a `Pipeline` with `StandardScaler`. Scaling is a good idea here because logistic regression is a regularized linear model and benefits from features being on comparable scales.


In [ ]:
X = df.drop(columns="is_safe")
y = df["is_safe"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y,
)

baseline_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=2000,
                random_state=1,
            ),
        ),
    ]
)

In [ ]:
baseline_pipeline.fit(X_train, y_train)

baseline_cv_f1 = cross_val_score(
    baseline_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="f1",
    n_jobs=-1,
).mean()

baseline_y_pred = baseline_pipeline.predict(X_test)
baseline_test_f1 = f1_score(y_test, baseline_y_pred)

print(f"Baseline CV F1: {baseline_cv_f1:.3f}")
print(f"Baseline test F1: {baseline_test_f1:.3f}")

A confusion matrix and classification report help us see how the baseline behaves beyond a single summary score.


In [ ]:
print(classification_report(y_test, baseline_y_pred, zero_division=0))

disp = ConfusionMatrixDisplay.from_predictions(y_test, baseline_y_pred, cmap="Blues")
plt.show()

## Tune the Model


We now tune a few logistic regression hyperparameters with `RandomizedSearchCV`. This is a practical choice when you want to search several combinations without paying the full cost of a large grid search.


In [ ]:
search_space = {
    "model__C": np.logspace(-3, 2, 20).tolist(),
    "model__class_weight": [None, "balanced"],
}

random_search = RandomizedSearchCV(
    estimator=baseline_pipeline,
    param_distributions=search_space,
    n_iter=8,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=1,
)

random_search.fit(X_train, y_train)

tuned_pipeline = random_search.best_estimator_
tuned_y_pred = tuned_pipeline.predict(X_test)
tuned_test_f1 = f1_score(y_test, tuned_y_pred)

print("Best parameters:", random_search.best_params_)
print(f"Best CV F1: {random_search.best_score_:.3f}")
print(f"Tuned test F1: {tuned_test_f1:.3f}")

A tuned model should also be interpreted, not only summarized with a single score. The classification report below shows how precision and recall changed after tuning.

In [ ]:
print(classification_report(y_test, tuned_y_pred, zero_division=0))

The tuned model can now be compared against the baseline side by side.


In [ ]:
comparison_df = pd.DataFrame(
    {
        "model": ["Baseline logistic regression", "Tuned logistic regression"],
        "cv_f1": [baseline_cv_f1, random_search.best_score_],
        "test_f1": [baseline_test_f1, tuned_test_f1],
    }
)

comparison_df

## Conclusion


This solution follows the full workflow from the exercise:

- choose a dataset,
- identify the problem type,
- build a baseline model,
- evaluate it with a sensible metric,
- tune hyperparameters,
- compare the baseline and tuned versions.

In this example, tuning improves the model slightly. That is a realistic outcome: hyperparameter tuning often helps, but the gain is not always dramatic.
